In [ ]:
# --- repo-root anchor (added during reorganisation) ---
# Walks up from the working directory to the repo root, so every path below
# resolves whether this notebook is run from code/notebooks/ or the repo root.
from pathlib import Path as _P
REPO_ROOT = _P.cwd().resolve()
while not (REPO_ROOT / 'requirements.txt').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR       = REPO_ROOT / 'data'
MODELS_DIR     = REPO_ROOT / 'models'
RESULTS_DIR    = REPO_ROOT / 'results'
REFERENCE_DIR  = REPO_ROOT / 'reference'
VALIDATION_DIR = RESULTS_DIR / 'validation'
# Staging area for freshly trained weights. Training ALWAYS writes here,
# never straight into models/, so released checkpoints are never overwritten.
CKPT_DIR       = MODELS_DIR / 'rbc_ckpts'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
print('repo root:', REPO_ROOT)


In [ ]:
import os, random
from typing import List, Tuple, Dict, Optional
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.functional as TF
from torchvision import transforms
from transformers import AutoImageProcessor

# ---------- CONFIG ----------
#ROOT_DIR = str(DATA_DIR / 'C dataset' / 'C in pair')
#ROOT_DIR = str(DATA_DIR / 'D_Reticulocyte')
ROOT_DIR = str(DATA_DIR / 'Paired dataset_10122025' / 'G')
NON_SICKLED_DIRNAME = "Non-sickled"
SICKLED_DIRNAME     = "Sickled"
IMAGE_EXTS = (".png", ".jpg")

# ViT preprocessing (works for any ViT)
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
IMG_SIZE = 224

# paired augmentations (same geometry for both images)
def paired_augment(img0: Image.Image, img1: Image.Image):
    # keep it light at first; you can add more once things work
    # random horizontal flip
    if random.random() < 0.5:
        img0 = TF.hflip(img0); img1 = TF.hflip(img1)
    # small rotation
    angle = random.uniform(-10, 10)
    img0 = TF.rotate(img0, angle, interpolation=transforms.InterpolationMode.BILINEAR)
    img1 = TF.rotate(img1, angle, interpolation=transforms.InterpolationMode.BILINEAR)
    return img0, img1

# to 3-channel + normalize exactly like ViT expects
to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # your crops look grayscale
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

# --------- PAIR DISCOVERY ---------
# Accept endings like "...frame0.png"/"...frame1.png"
PAIR_SUFFIXES = [
    ("frame0", "frame1"),
]

def split_pair_key(fname: str) -> Optional[Tuple[str, int]]:
    """Return (base_key, idx) where idx∈{0,1} if name matches a known pattern, else None."""
    stem, ext = os.path.splitext(fname)
    for s0, s1 in PAIR_SUFFIXES:
        if stem.endswith(s0):
            return stem[:-len(s0)], 0
        if stem.endswith(s1):
            return stem[:-len(s1)], 1
    return None

def find_pairs(folder: str) -> Dict[str, Dict[int, str]]:
    """
    Scan folder and return {pair_id: {0: before_path, 1: after_path}} for files that have both sides.
    """
    bucket: Dict[str, Dict[int, str]] = {}
    for f in os.listdir(folder):
        if not f.lower().endswith(IMAGE_EXTS): 
            continue
        parsed = split_pair_key(f)
        if parsed is None:
            continue
        key, idx = parsed
        d = bucket.setdefault(key, {})
        d[idx] = os.path.join(folder, f)

    # keep only complete pairs
    complete = {k: v for k, v in bucket.items() if 0 in v and 1 in v}
    return complete

# --------- DATASET ---------
class PairedCellDataset(Dataset):
    """
    Each item: (tensor_before, tensor_after, label)
    label = 0 for Non-sickled C (unchanged), 1 for Sickled C (changed)
    """
    def __init__(self, root_dir: str, augment: bool = False):
        non_path = os.path.join(root_dir, NON_SICKLED_DIRNAME)
        sic_path = os.path.join(root_dir, SICKLED_DIRNAME)

        non_pairs = find_pairs(non_path)  # label 0
        sic_pairs = find_pairs(sic_path)  # label 1

        self.samples: List[Tuple[str, str, int]] = []
        for key, d in non_pairs.items():
            self.samples.append((d[0], d[1], 0))
        for key, d in sic_pairs.items():
            self.samples.append((d[0], d[1], 1))

        self.augment = augment

    def __len__(self): 
        return len(self.samples)

    def __getitem__(self, idx):
        p0, p1, y = self.samples[idx]
        img0 = Image.open(p0).convert("RGB")
        img1 = Image.open(p1).convert("RGB")

        if self.augment:
            img0, img1 = paired_augment(img0, img1)

        x0 = to_tensor(img0)
        x1 = to_tensor(img1)
        y  = torch.tensor(y, dtype=torch.float32)  # BCEWithLogitsLoss wants float

        return x0, x1, y

# --------- SPLIT + LOADERS ---------
def make_loaders(root_dir=ROOT_DIR, batch_size=32, val_ratio=0.2, seed=42, augment_train=True):
    ds_all = PairedCellDataset(root_dir, augment=False)  # build index once

    # stratified split by label to keep class balance
    labels = [int(y) for (_, _, y) in ds_all.samples]
    idxs0 = [i for i, y in enumerate(labels) if y == 0]
    idxs1 = [i for i, y in enumerate(labels) if y == 1]

    def split_idxs(idxs):
        random.Random(seed).shuffle(idxs)
        n_val = max(1, int(len(idxs) * val_ratio))
        return idxs[n_val:], idxs[:n_val]

    tr0, va0 = split_idxs(idxs0)
    tr1, va1 = split_idxs(idxs1)
    train_idx = tr0 + tr1
    val_idx   = va0 + va1
    random.Random(seed).shuffle(train_idx)
    random.Random(seed).shuffle(val_idx)

    # two real datasets so train can have augmentations if desired
    train_ds = PairedCellDataset(root_dir, augment=augment_train)
    val_ds   = PairedCellDataset(root_dir, augment=False)

    train_subset = Subset(train_ds, train_idx)
    val_subset   = Subset(val_ds,   val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

    print(f"Total pairs: {len(ds_all)} | Train: {len(train_subset)} | Val: {len(val_subset)}")
    print(f"Class balance (train): changed={sum(int(train_ds.samples[i][2]) for i in train_idx)}, "
          f"unchanged={sum(1-int(train_ds.samples[i][2]) for i in train_idx)}")
    print(f"Class balance (val):   changed={sum(int(val_ds.samples[i][2]) for i in val_idx)}, "
          f"unchanged={sum(1-int(val_ds.samples[i][2]) for i in val_idx)}")
    return train_loader, val_loader

# ---------- USAGE ----------
# train_loader, val_loader = make_loaders(batch_size=32, val_ratio=0.2, seed=42, augment_train=True)


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 33288.13it/s]
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 19972.88it/s]


In [10]:
import torch, torch.nn as nn
from transformers import ViTModel
import torch.nn.functional as F

class SiameseViTChange(nn.Module):
    def __init__(self, backbone="google/vit-base-patch16-224-in21k", proj_dim=512, dropout=0.1):
        super().__init__()
        self.vit = ViTModel.from_pretrained(backbone)
        h = self.vit.config.hidden_size          # 768 for ViT-Base
        self.proj = nn.Sequential(
            nn.Linear(h, proj_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.head = nn.Sequential(
            nn.Linear(proj_dim*2, proj_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(proj_dim, 1)            # BCEWithLogitsLoss
        )

    def encode(self, x):                       # x: (B,3,224,224), normalized
        # out = self.vit(pixel_values=x).pooler_output
        # return self.proj(out)

        # Ensure contiguity
        x = x.contiguous()
        out = self.vit(pixel_values=x)
        cls = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0]
        cls = cls.contiguous()
        proj = self.proj(cls).contiguous()
        return proj

    def forward(self, x0, x1):
        # f0, f1 = self.encode(x0), self.encode(x1)
        # z = torch.cat([ (f0 - f1).abs(), f0 * f1 ], dim=1)
        # logit = self.head(z).squeeze(1)
        # return logit
        
        # Ensure contiguity
        f0, f1 = self.encode(x0), self.encode(x1)
        # (optional but recommended) normalize embeddings
        f0 = F.normalize(f0, dim=1)
        f1 = F.normalize(f1, dim=1)

        z = torch.cat([(f0 - f1).abs(), f0 * f1], dim=1)
        logit = self.head(z)
        return logit.reshape(-1)


In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Tuple
from sklearn.metrics import roc_auc_score

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
    amp_enabled = True
    scaler = torch.cuda.amp.GradScaler("cuda")
    # torch.backends.cudnn.benchmark = True            # speed if shapes are stable
    # torch.set_float32_matmul_precision("high")
    # # Prefer BF16 if supported (more stable than FP16)
    # amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16       # allow TF32 on Ampere/ADA
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
#     amp_enabled = True
#     scaler = None                                    # no GradScaler on MPS
else:
    device = torch.device("cpu")
    amp_enabled = False
    scaler = None

print("Device:", device)

# # 1) Device (MPS on Apple Silicon, else CUDA/CPU)
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")
# print("Device:", device)

# 2) Build loaders (uses your make_loaders)
train_loader, val_loader = make_loaders(
    root_dir=ROOT_DIR,
    batch_size=32,
    val_ratio=0.2,
    seed=42,
    augment_train=True
)

# 3) Build model (uses your SiameseViTChange)
model = SiameseViTChange(backbone="google/vit-base-patch16-224-in21k", proj_dim=512, dropout=0.1).to(device)

# 4) Class imbalance handling -> pos_weight for BCEWithLogitsLoss
#    We can estimate from the training subset labels:
def count_labels(loader: DataLoader) -> Tuple[int,int]:
    pos = neg = 0
    for _, _, y in loader.dataset:  # uses Subset -> iterates underlying dataset safely
        if int(y.item()) == 1: pos += 1
        else: neg += 1
    return pos, neg

pos, neg = count_labels(train_loader)
print(f"Train label counts -> changed (1): {pos} | unchanged (0): {neg}")
pos_weight_val = (neg / max(1, pos)) if pos > 0 else 1.0
pos_weight = torch.tensor([pos_weight_val], device=device, dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 5) Optimizer / (optional) warmup freeze
optimizer = torch.optim.AdamW([
    {"params": model.proj.parameters(), "lr": 1e-3},
    {"params": model.head.parameters(), "lr": 1e-3},
    {"params": model.vit.parameters(), "lr": 2e-5},
], weight_decay=0.01)

# Optional: start by freezing ViT for 1–2 epochs then unfreeze
freeze_warmup_epochs = 1
for p in model.vit.parameters(): p.requires_grad = False

def step_epoch(loader, train: bool):
    model.train(train)
    total_loss, n = 0.0, 0
    correct, total = 0, 0
    all_logits, all_labels = [], []
    t0 = time.time()
    for x0, x1, y in loader:
        x0, x1, y = x0.to(device), x1.to(device), y.to(device)
        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            logits = model(x0, x1)
            # Flatten and ensure contiguity for both logits and y
            logits = logits.reshape(-1)
            y = y.reshape(-1).float()
            loss = criterion(logits, y)

        if train:
            if scaler is not None:  # CUDA
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:                   # MPS/CPU
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x0.size(0)
        n += x0.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()
        correct += (preds.squeeze().cpu() == y.long().cpu()).sum().item()
        total += y.size(0)

        all_logits.append(logits.detach().cpu())
        all_labels.append(y.detach().cpu())

    # metrics
    import numpy as np
    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy().astype(int)
    probs_np  = 1.0/(1.0+np.exp(-logits_np))
    # F1 at 0.5
    tp = ((probs_np>=0.5) & (labels_np==1)).sum()
    fp = ((probs_np>=0.5) & (labels_np==0)).sum()
    fn = ((probs_np<0.5) & (labels_np==1)).sum()
    precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall    = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0

    # Optional ROC-AUC if sklearn available
    auc = roc_auc_score(labels_np, probs_np)

    return {
        "loss": total_loss/max(1,n),
        "acc": correct/max(1,total),
        "f1": f1,
        "auc": auc,
        "time_s": time.time()-t0
    }

# 7) Train
best_auc, best_path = 0.0, str(CKPT_DIR / 'siamese_vit_ISC_Haolin.pt')
epochs = 12

for epoch in range(1, epochs+1):
    # Unfreeze after warmup
    if epoch == freeze_warmup_epochs+1:
        for p in model.vit.parameters(): p.requires_grad = True
        # lower LR for the head to keep stable after unfreezing
        for g in optimizer.param_groups:
            if g["lr"] >= 1e-3: g["lr"] = 5e-4

    tr = step_epoch(train_loader, train=True)
    va = step_epoch(val_loader,   train=False)

    print(f"[E{epoch:02d}] "
          f"train: loss={tr['loss']:.4f}, acc={tr['acc']:.3f}, f1={tr['f1']:.3f}, auc={tr['auc']:.3f}, {tr['time_s']:.1f}s | "
          f"val: loss={va['loss']:.4f}, acc={va['acc']:.3f}, f1={va['f1']:.3f}, auc={va['auc']:.3f}, {va['time_s']:.1f}s")

    # save epoch-10 checkpoint (tune if you think specific epoch is best):
    if epoch == 10:
        torch.save(model.state_dict(), str(CKPT_DIR / 'siamese_vit_ISC_epoch10.pt'))
        print("  ↳ saved epoch-10 checkpoint to siamese_vit_ISC_epoch10.pt")

    # save best by AUC (ignore NaN)
    if va["auc"] > best_auc:
        best_auc = va["auc"]
        torch.save(model.state_dict(), best_path)
        print(f"  ↳ saved best to {best_path} (AUC={best_auc:.3f})")

print("Done. Best AUC:", best_auc)



Device: cpu
Total pairs: 320 | Train: 256 | Val: 64
Class balance (train): changed=216, unchanged=40
Class balance (val):   changed=54, unchanged=10
Train label counts -> changed (1): 216 | unchanged (0): 40
[E01] train: loss=0.2151, acc=0.844, f1=0.915, auc=0.681, 23.1s | val: loss=0.2135, acc=0.844, f1=0.915, auc=0.730, 5.6s
  ↳ saved best to siamese_vit_c_Haolin.pt (AUC=0.730)
[E02] train: loss=0.2042, acc=0.844, f1=0.915, auc=0.879, 80.1s | val: loss=0.1948, acc=0.828, f1=0.906, auc=0.930, 8.4s
  ↳ saved best to siamese_vit_c_Haolin.pt (AUC=0.930)
[E03] train: loss=0.1820, acc=0.938, f1=0.964, auc=0.937, 73.7s | val: loss=0.1794, acc=0.797, f1=0.871, auc=0.887, 8.6s
[E04] train: loss=0.1622, acc=0.934, f1=0.961, auc=0.954, 72.0s | val: loss=0.1518, acc=0.859, f1=0.914, auc=0.935, 8.9s
  ↳ saved best to siamese_vit_c_Haolin.pt (AUC=0.935)
[E05] train: loss=0.1366, acc=0.953, f1=0.972, auc=0.983, 73.6s | val: loss=0.1329, acc=0.891, f1=0.932, auc=0.935, 8.9s
[E06] train: loss=0.1091,

In [10]:
# ===================== Reload + Threshold Optimization =====================
import torch, numpy as np
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, classification_report, precision_recall_curve, roc_curve
import matplotlib.pyplot as plt

# ---------- CONFIG ----------
CKPT_PATH = str(MODELS_DIR / 'siamese_vit_c_Haolin.pt')   # <- set this to your weights file
POS_LABEL = 1                             # which class id means "YES / sickled / changed"
METRIC = "f1"                             # choose: "f1", "youden", "balanced_acc"
TH_GRID = np.linspace(0.01, 0.99, 99)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Expect 'model' defined in earlier cells, and either val_loader or test_loader available
assert "model" in globals(), "Expected a 'model' from earlier cells."
loader = globals().get("val_loader", None) or globals().get("test_loader", None)
assert loader is not None, "Need a DataLoader named 'val_loader' or 'test_loader'."

# ---------- Reload weights ----------
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
state_dict = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing or unexpected:
    print("[Warn] load_state_dict mismatches:",
          "\n  missing:", missing, "\n  unexpected:", unexpected)
model.to(device).eval()

# ---------- Convert logits -> P(positive) ----------
def logits_to_posprob(logits, pos_label=1):
    if logits.ndim == 1 or logits.shape[-1] == 1:     # BCE logit
        return torch.sigmoid(logits.reshape(-1)).detach().cpu().numpy()
    elif logits.shape[-1] == 2:                       # 2-class softmax
        return torch.softmax(logits, dim=-1)[:, pos_label].detach().cpu().numpy()
    else:
        raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")

# ---------- Collect probabilities & labels ----------
all_probs, all_labels = [], []
with torch.no_grad():
    for batch in loader:
        # supports (img1, img2, y) or (x, y)
        if isinstance(batch, (list, tuple)):
            if len(batch) == 3:
                a, b, y = batch
                a = a.to(device); b = b.to(device)
                logits = model(a, b)
            elif len(batch) == 2:
                x, y = batch
                x = x.to(device)
                logits = model(x)
            else:
                raise ValueError("Unexpected batch format; expected (img1,img2,y) or (x,y).")
        else:
            raise ValueError("Unexpected batch type from DataLoader.")
        all_probs.append(logits_to_posprob(logits, pos_label=POS_LABEL))
        all_labels.append(y.detach().cpu().numpy())

probs = np.concatenate(all_probs)
labels = np.concatenate(all_labels).astype(int)

# # ---------- Sweep thresholds ----------
# def eval_grid(probs, y_true, th_grid, pos_label=1, metric="f1"):
#     rows = []
#     for th in th_grid:
#         y_pred = (probs >= th).astype(int)
#         p, r, f1, _ = precision_recall_fscore_support(
#             y_true, y_pred, pos_label=pos_label, average="binary", zero_division=0
#         )
#         tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
#         tpr = tp / (tp + fn + 1e-9)
#         tnr = tn / (tn + fp + 1e-9)
#         youden = tpr - (1 - tnr)
#         bacc = 0.5 * (tpr + tnr)
#         rows.append((th, p, r, f1, youden, bacc))
#     M = np.array(rows)  # th, P, R, F1, YJ, BACC
#     col = {"f1":3, "youden":4, "balanced_acc":5}[metric]
#     best_idx = int(np.argmax(M[:, col]))
#     return M, best_idx

# grid_stats, best_idx = eval_grid(probs, labels, TH_GRID, pos_label=POS_LABEL, metric=METRIC)
# best_th, P, R, F1, YJ, BACC = grid_stats[best_idx]
# auroc = roc_auc_score(labels, probs)
# auprc = average_precision_score(labels, probs, pos_label=POS_LABEL)

# # ---------- Print results ----------
# print(f"\nBest threshold by {METRIC}: {best_th:.4f}")
# print(f"Precision={P:.3f}  Recall={R:.3f}  F1={F1:.3f}  YoudenJ={YJ:.3f}  BalancedAcc={BACC:.3f}")
# print(f"AUROC={auroc:.3f}  AUPRC={auprc:.3f}")

# # Also print confusion matrix & report at best threshold:
# y_pred = (probs >= best_th).astype(int)
# print("\nConfusion Matrix (at best threshold):")
# print(confusion_matrix(labels, y_pred, labels=[0,1]))
# print("\nClassification Report (at best threshold):")
# print(classification_report(labels, y_pred, zero_division=0, target_names=["NO","YES"]))

# # Keep it available for later cells if you want to reuse it
# BEST_THRESHOLD = float(best_th)
# print(f"\nBEST_THRESHOLD = {BEST_THRESHOLD:.6f}")

# ---------- Alternative: Direct target precision / FPR constraints ----------
# ----------------- CONFIG -----------------
P_MIN = 0.98    # target precision threshold
FPR_MAX = 0.01  # target maximum false positive rate

# ---------- Precision ≥ target (pick max recall among feasible) ----------
prec, rec, thr_pr = precision_recall_curve(labels, probs, pos_label=POS_LABEL)
mask = prec[1:] >= P_MIN                               # align thresholds with prec[1:]
if np.any(mask):
    # choose the feasible threshold with the highest recall
    i = int(np.argmax(rec[1:][mask]))
    th_prec = float(thr_pr[mask][i])
    y_pred = (probs >= th_prec).astype(int)
    P, R, F1, _ = precision_recall_fscore_support(labels, y_pred, pos_label=POS_LABEL,
                                                  average="binary", zero_division=0)
    print(f"[Precision≥{P_MIN:.2f}] threshold={th_prec:.6f}  P={P:.3f} R={R:.3f} F1={F1:.3f}")
else:
    print(f"No threshold reaches precision ≥ {P_MIN:.2f} on this set.")

# ---------- FPR ≤ target (ignore inf; pick max TPR among feasible) ----------
fpr, tpr, thr_roc = roc_curve(labels, probs, pos_label=POS_LABEL)
mask = (fpr <= FPR_MAX) & np.isfinite(thr_roc)
if np.any(mask):
    sub_fpr, sub_tpr, sub_thr = fpr[mask], tpr[mask], thr_roc[mask]
    j = int(np.argmax(sub_tpr))                        # best recall under FPR constraint
    th_fpr = float(sub_thr[j])
    y_pred = (probs >= th_fpr).astype(int)
    P, R, F1, _ = precision_recall_fscore_support(labels, y_pred, pos_label=POS_LABEL,
                                                  average="binary", zero_division=0)
    print(f"[FPR≤{FPR_MAX:.3f}] threshold={th_fpr:.6f}  P={P:.3f} R={R:.3f} F1={F1:.3f}  Specificity≈{1-sub_fpr[j]:.3f}")
else:
    # nearest achievable point
    k = int(np.argmin(np.maximum(fpr - FPR_MAX, 0)))
    print(f"No threshold achieves FPR ≤ {FPR_MAX:.3f}. Nearest FPR={fpr[k]:.3f} at threshold={thr_roc[k]:.6f}")

# ---------- F0.5 max using *all* meaningful cut points ----------
# build candidate thresholds from unique scores, their midpoints, and PR/ROC thresholds
u = np.unique(probs)
mid = (u[1:] + u[:-1]) / 2.0
candidates = np.unique(
    np.concatenate([
        [0.0, 1.0],
        u,
        mid,
        thr_pr[np.isfinite(thr_pr)],
        thr_roc[np.isfinite(thr_roc) & ~np.isinf(thr_roc)],
    ])
)
candidates = candidates[(candidates >= 0.0) & (candidates <= 1.0)]

def metrics_at(th, beta=0.5):
    y = (probs >= th).astype(int)
    p, r, _, _ = precision_recall_fscore_support(labels, y, pos_label=POS_LABEL,
                                                 average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, y, labels=[0,1]).ravel()
    fpr = fp / (fp + tn + 1e-9)
    b2 = beta * beta
    fbeta = (1 + b2) * p * r / (b2 * p + r + 1e-12)
    return fbeta, p, r, fpr

vals = np.array([metrics_at(t, beta=0.5) for t in candidates])   # cols: F0.5, P, R, FPR
idx = int(np.argmax(vals[:, 0]))
th_f05 = float(candidates[idx])
F05, P, R, FPRv = vals[idx]
print(f"[F0.5 max] threshold={th_f05:.6f}  F0.5={F05:.3f}  P={P:.3f}  R={R:.3f}  FPR={FPRv:.3f}")

# (Optional) show top-5 thresholds by F0.5 to see if there’s a plateau
order = np.argsort(-vals[:, 0])
for rank in range(min(5, len(order))):
    i = order[rank]
    print(f"  #{rank+1}: th={candidates[i]:.6f}  F0.5={vals[i,0]:.3f}  P={vals[i,1]:.3f}  R={vals[i,2]:.3f}  FPR={vals[i,3]:.3f}")


[Precision≥0.98] threshold=0.497272  P=0.963 R=0.981 F1=0.972
[FPR≤0.010] threshold=0.821673  P=1.000 R=0.849 F1=0.918  Specificity≈1.000
[F0.5 max] threshold=0.505160  F0.5=0.981  P=0.981  R=0.981  FPR=0.042
  #1: th=0.505160  F0.5=0.981  P=0.981  R=0.981  FPR=0.042
  #2: th=0.513047  F0.5=0.981  P=0.981  R=0.981  FPR=0.042
  #3: th=0.519132  F0.5=0.977  P=0.981  R=0.962  FPR=0.042
  #4: th=0.525217  F0.5=0.977  P=0.981  R=0.962  FPR=0.042
  #5: th=0.587984  F0.5=0.973  P=0.980  R=0.943  FPR=0.042
